# 01 · Entendimento do negócio e dos dados (v4)
**CRISP-DM:** Entendimento do Negócio + Entendimento dos Dados

## 1. Entendimento do negócio
**Problema.** Quando um incidente é aberto, o operador precisa saber se as APIs (recursos) envolvidas estão operando dentro de um **regime normal** e, se não estão, **qual indicador saiu do padrão**: erros de negócio (4xx), erros de sistema (5xx) ou latência. O comportamento normal de uma API muda com o horário, o dia e os processamentos em lote, e por isso não basta um limite fixo.

**Decisão apoiada.** *O recurso estava em regime normal no momento do registro do incidente? Se não, qual indicador foi impactado e desde quando?* Como informação complementar: o impacto é **generalizado** ou concentrado em um **consumidor específico** capaz de alterar o resultado agregado?

**Objetivo de mineração.** Para cada recurso: (1) caracterizar seus regimes normais a partir dos Access Logs agregados; (2) no instante do registro de um incidente, classificar o período imediatamente anterior como **Normal** ou **Residual**; (3) apontar o **indicador responsável** e estimar o **início** do desvio.

**Questões de pesquisa.**
- **RQ1:** Access Logs agregados permitem caracterizar os regimes normais de cada recurso e decidir, no instante do registro, se o recurso está em regime normal?
- **RQ2:** Quando o recurso não está normal, o método identifica corretamente o indicador responsável (4xx, 5xx, latência) e o início do desvio?
- **RQ3:** Qual o menor período antes do registro (15, 30 ou 60 min) que já permite essa decisão, e como o atraso entre a degradação e o registro afeta o resultado?
- **RQ4:** Um histórico de 3 meses produz um perfil melhor que o do último mês?

**Critérios de sucesso** (definidos antes da modelagem):

| Critério | Meta |
|---|---|
| Consultas de controle (sem falha) com saída Normal | ≥ 95% |
| Sensibilidade no registro, faixa de 15 min, intensidade média ou alta | ≥ 70% |
| Acurácia do indicador responsável (casos detectados) | ≥ 90% |
| Início real dentro da zona estimada | ≥ 70% |
| Sensibilidade do método proposto ≥ baselines (McNemar) | sem perda significativa |

In [1]:
%run ./00_funcoes.ipynb

00_funcoes (v4) carregado · /home/wsl/gh/mestrado/claude9/v4


## 2. Entendimento dos dados
**Cada linha da base é um agregado por minuto × recurso × código HTTP × cliente**: `QT_TTL_TRAN` é o total de execuções naquele minuto e `QT_TMP_MEDI_EXEA`, o tempo médio de execução delas. Não há registros por requisição; por isso, percentis de latência são aproximados a partir dessas médias por minuto (notebook 02). `TS_EVT` está em UTC e é convertido para o horário de Brasília. `CD_CLI` vem anonimizado (SHA-256 com semente).

### Extração e seleção dos recursos
O arquivo `APIs-NovoCj` (13 GB, 118 milhões de linhas) é lido em blocos. Ficam só os recursos com dados em **todos os dias de 01/06 a 20/09** (horário de Brasília); os demais são descartados. O recorte usado é de 01/06 a 20/09: histórico de junho a agosto e teste em setembro.

In [2]:
import time
t0 = time.time()
todos, dias = extrair()
print(f"extraídos {len(todos):,} registros (01/06–20/09) em {time.time() - t0:.0f} s")
dias_rec = dias[(dias["dia"] >= PERIODO_DADOS[0]) & (dias["dia"] < PERIODO_DADOS[1])].groupby("recurso")["dia"].nunique()
n_dias = (PERIODO_DADOS[1] - PERIODO_DADOS[0]).days
selecao = pd.DataFrame({"dias_com_dados": dias_rec, "dias_possiveis": n_dias,
                        "primeiro_dia_arquivo": dias.groupby("recurso")["dia"].min(),
                        "volume_recorte": todos.groupby("recurso")["qtd"].sum()})
selecao["completo"] = selecao["dias_com_dados"] == n_dias
print(f"recursos no arquivo: {dias['recurso'].nunique()} · completos: {int(selecao['completo'].sum())}")
selecao.sort_values("volume_recorte", ascending=False)

extraídos 91,786,326 registros (01/06–20/09) em 112 s


recursos no arquivo: 18 · completos: 14


,dias_com_dados,dias_possiveis,primeiro_dia_arquivo,volume_recorte,completo
recurso,,,,,
18323653,112,112,2026-05-19,956303660,True
18323648,112,112,2026-05-19,900981871,True
8175425,105,112,2026-04-30,572873536,False
18323650,112,112,2026-05-19,462985817,True
9155284,112,112,2026-04-30,385612865,True
8175405,105,112,2026-04-30,301110672,False
8180434,112,112,2026-04-30,191140988,True
8180412,112,112,2026-04-30,182992106,True
18323651,112,112,2026-05-19,151328356,True


In [3]:
recursos_ok = selecao.index[selecao["completo"]]
bruto = todos[todos["recurso"].isin(recursos_ok)].reset_index(drop=True)
del todos
bruto.to_parquet(ARQUIVO_DADOS, index=False)
selecao.to_csv(RESULTADOS / "01_selecao_recursos.csv")
print(f"extrato salvo: {len(bruto):,} linhas · {bruto['recurso'].nunique()} recursos")
print(f"volume dos descartados: {selecao.loc[~selecao['completo'], 'volume_recorte'].sum() / selecao['volume_recorte'].sum():.1%} do recorte")

extrato salvo: 24,495,617 linhas · 14 recursos
volume dos descartados: 21.0% do recorte


### Códigos dos recursos
No artigo, os recursos são identificados por **A01–A14**, em ordem decrescente de volume (para não confundir com os E01–E12 das versões anteriores, que são outros recursos). O único recurso em comum com a base anterior (`18169917`, o E01 das versões anteriores) é identificado abaixo. O mapeamento fica só nos resultados.

In [4]:
vol = bruto.groupby("recurso")["qtd"].sum().sort_values(ascending=False)
mapa = {int(r): f"A{i + 1:02d}" for i, r in enumerate(vol.index)}
arquivo_codigos().write_text(json.dumps(mapa, indent=2))
print("recurso em comum com as versões anteriores:", {r: c for r, c in mapa.items() if r == 18169917})
pd.Series(mapa, name="código")

recurso em comum com as versões anteriores: {18169917: 'A10'}


18323653    A01
18323648    A02
18323650    A03
9155284     A04
8180434     A05
8180412     A06
18323651    A07
8190019     A08
18139261    A09
18169917    A10
8617320     A11
8179198     A12
15945008    A13
18170590    A14
Name: código, dtype: str

In [5]:
print(f"linhas: {len(bruto):,}")
print("período (Brasília):", bruto["ts"].min(), "→", bruto["ts"].max())
print("recursos:", bruto["recurso"].nunique(), "· clientes:", bruto["cliente"].nunique())
print("status:", sorted(bruto["status"].unique()))
print("nulos:", int(bruto.isna().sum().sum()), "· qtd ≤ 0:", int((bruto["qtd"] <= 0).sum()), "· tmp < 0:", int((bruto["tmp"] < 0).sum()))

linhas: 24,495,617


período (Brasília): 2026-06-01 00:00:00 → 2026-09-20 20:59:00
recursos: 14 · clientes: 5506


status: [np.int16(0), np.int16(200), np.int16(400), np.int16(401), np.int16(403), np.int16(404), np.int16(415), np.int16(422), np.int16(423), np.int16(499), np.int16(500), np.int16(502), np.int16(503), np.int16(504)]


nulos: 0 · qtd ≤ 0: 0 · tmp < 0: 0


### Volume médio diário e taxa de 4xx por mês
Mudanças de patamar entre meses (crescimento de volume, mudança da taxa de erros de negócio) motivam a comparação de histórico de 3 meses × 1 mês.

In [6]:
mes = bruto["ts"].dt.to_period("M")
vol_dia = bruto.groupby(["recurso", mes])["qtd"].sum() / bruto.groupby(["recurso", mes])["ts"].apply(lambda s: s.dt.date.nunique())
vol_dia = (vol_dia.unstack() / 1000).round(1)
vol_dia.index = vol_dia.index.map(rot)
t4 = bruto[bruto["status"] // 100 == 4].groupby(["recurso", mes])["qtd"].sum() / bruto.groupby(["recurso", mes])["qtd"].sum()
t4 = t4.unstack().round(3)
t4.index = t4.index.map(rot)
display(vol_dia.sort_index())
display(t4.sort_index())
vol_dia.to_csv(RESULTADOS / "01_volume_diario_mes.csv")
t4.to_csv(RESULTADOS / "01_taxa4xx_mes.csv")

ts,2026-06,2026-07,2026-08,2026-09
recurso,,,,
A01,8202.0,8573.2,8658.0,8803.7
A02,7637.6,7978.8,8187.5,8535.0
A03,3967.9,4191.4,4206.3,4181.0
A04,3549.1,3581.3,3308.5,3277.8
A05,1638.1,1931.8,1687.6,1489.8
A06,1685.1,1303.3,1895.5,1663.8
A07,1667.4,1386.6,1146.7,1138.7
A08,343.0,376.6,483.3,751.3
A09,430.5,465.4,444.7,539.2


ts,2026-06,2026-07,2026-08,2026-09
recurso,,,,
A01,0.005,0.002,0.001,0.001
A02,0.008,0.011,0.013,0.007
A03,0.016,0.006,0.005,0.005
A04,0.009,0.006,0.006,0.004
A05,0.190,0.160,0.178,0.217
A06,0.672,0.574,0.493,0.474
A07,0.085,0.008,0.006,0.007
A08,0.079,0.069,0.229,0.449
A09,0.008,0.006,0.005,0.005


### Chaves repetidas

In [7]:
chave = ["ts", "recurso", "status", "cliente"]
rep = bruto.duplicated(chave, keep=False)
qualidade = dict(
    linhas=len(bruto), linhas_em_chave_repetida=int(rep.sum()), linhas_excedentes=int(bruto.duplicated(chave).sum()),
    frac_volume_em_chave_repetida=float(bruto.loc[rep, "qtd"].sum() / bruto["qtd"].sum()),
    iguais_nas_metricas=int(bruto[rep].duplicated(chave + ["qtd", "tmp"]).sum()))
qualidade

{'linhas': 24495617,
 'linhas_em_chave_repetida': 675529,
 'linhas_excedentes': 375697,
 'frac_volume_em_chave_repetida': 0.075348088150542,
 'iguais_nas_metricas': 113796}

### Cobertura por recurso

In [8]:
cob = bruto.groupby("recurso").agg(inicio=("ts", "min"), fim=("ts", "max"), minutos=("ts", "nunique"),
                                   clientes=("cliente", "nunique"), volume=("qtd", "sum"))
cob["cobertura_min"] = cob["minutos"] / (((cob["fim"] - cob["inicio"]) / pd.Timedelta("1min")) + 1)
cob["part_volume"] = cob["volume"] / cob["volume"].sum()
cob["transacoes_por_15min_mediana"] = bruto.assign(j=bruto["ts"].dt.floor("15min")).groupby(["recurso", "j"])["qtd"].sum().groupby("recurso").median()
cob.index = cob.index.map(mapa)
cob = cob.sort_index()
cob

,inicio,fim,minutos,clientes,volume,cobertura_min,part_volume,transacoes_por_15min_mediana
recurso,,,,,,,,
A01,2026-06-01,2026-09-20 20:59:00,160977,36,956303660,0.999236,0.275996,85335.0
A02,2026-06-01,2026-09-20 20:59:00,160977,45,900981871,0.999236,0.260030,67001.0
A03,2026-06-01,2026-09-20 20:59:00,160973,46,462985817,0.999212,0.133621,42996.5
A04,2026-06-01,2026-09-20 20:59:00,160958,45,385612865,0.999119,0.111291,34148.0
A05,2026-06-01,2026-09-20 17:49:00,115143,43,191140988,0.715574,0.055165,1906.0
A06,2026-06-01,2026-09-20 20:59:00,123386,21,182992106,0.765897,0.052813,14839.5
A07,2026-06-01,2026-09-20 20:59:00,160896,29,151328356,0.998734,0.043674,11646.5
A08,2026-06-01,2026-09-20 20:59:00,161097,5390,51974064,0.999981,0.015000,3999.0
A09,2026-06-01,2026-09-20 20:59:00,160860,35,51910179,0.998510,0.014982,1617.0


### Proporção por família HTTP e latência

In [9]:
fam = bruto.groupby(bruto["status"] // 100)["qtd"].sum()
display((fam / fam.sum()).round(4))
desc = bruto[["qtd", "tmp"]].describe(percentiles=[0.5, 0.9, 0.99]).T
extremos = dict(latencia_acima_60s=int((bruto["tmp"] > 60_000).sum()), latencia_zero=int((bruto["tmp"] == 0).sum()))
display(desc)
extremos

status
0    0.0000
2    0.9410
4    0.0581
5    0.0009
Name: qtd, dtype: float64

,count,mean,std,min,50%,90%,99%,max
qtd,24495617.0,141.450671,591.707752,1.0,5.0,210.0,2011.0,15065.0
tmp,24495617.0,94.284727,810.551177,0.0,42.0,87.0,294.0,181183.0


{'latencia_acima_60s': 744, 'latencia_zero': 63097}

## 3. Regras de validade
1. Conversão de `TS_EVT` de UTC para America/Sao_Paulo antes de qualquer corte temporal.
2. Consolidação das chaves repetidas (soma de volume, média de latência ponderada).
3. Latências extremas mantidas (estatísticas robustas).
4. Só entram recursos com dados em todos os dias de 01/06 a 20/09.
5. Linhas com código HTTP 0 são mantidas (entram no volume, não nas taxas de 4xx/5xx).

In [10]:
qualidade.update(extremos, periodo_inicio=bruto["ts"].min(), periodo_fim=bruto["ts"].max(),
                 recursos=int(bruto["recurso"].nunique()), clientes=int(bruto["cliente"].nunique()),
                 familias={int(k): float(v) for k, v in (fam / fam.sum()).items()})
salvar_json(qualidade, "01_qualidade.json")
cob.to_csv(RESULTADOS / "01_cobertura_recursos.csv")